In [1]:
import pandas as pd
import json
import re

In [2]:
def clean(text):
    """Remove messy whitespace and escape chars."""
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', re.sub(r'\\r\\n', '; ', str(text))).strip()

In [3]:
def build_profile(row):
    """
    Combine summary (weighted 2x) + article/proceeding titles.
    Summary is repeated to give it higher importance in embeddings.
    """
    parts = []

    # Summary x2 = higher semantic weight
    summary = clean(row['summary'])
    if summary:
        parts.append(summary)
        parts.append(summary)  # repeat for emphasis

    # Add publication titles (these contain domain keywords)
    for col in ['articles', 'proceedings']:
        pub_text = clean(row[col])
        if pub_text:
            # Take first 600 chars to avoid overly long profiles
            parts.append(pub_text[:600])

    return ' '.join(parts)

In [4]:
df = pd.read_csv('all_lecturers.csv')
print(f"Loaded {len(df)} lecturers")

Loaded 69 lecturers


In [5]:
df['profile'] = df.apply(build_profile, axis=1)

In [6]:
df_clean = df[df['profile'].str.strip() != ''].copy()
df_clean = df_clean.reset_index(drop=True)
print(f"After cleaning: {len(df_clean)} lecturers with profiles")

After cleaning: 65 lecturers with profiles


In [8]:
df_clean.to_csv('lecturers_clean_2.csv', index=False)
print("✅ Saved: lecturers_clean_2.csv")

✅ Saved: lecturers_clean_2.csv


In [9]:
print("\n--- Sample profiles ---")
for _, row in df_clean.head(3).iterrows():
    print(f"\n👤 {row['name']}")
    print(f"   Profile (first 200 chars): {row['profile'][:200]}...")


--- Sample profiles ---

👤 Ts. Abdul Rahman bin Mat
   Profile (first 200 chars): Abdul Rahman Bin Mat is a senior lecturer for Software Engineering programme at the Faculty of Computer Science & Information Technology, Universiti Malaysia Sarawak (UNIMAS). He received his Masters ...

👤 Prof Madya Ts Dr Adnan Shahid Khan
   Profile (first 200 chars): Adnan Shahid Khan is currently working as Assoc. Professor at Faculty of Computer Science and Information Technology, Universiti Malaysia Sarawak. He has completed his Postdoctoral, Ph.D. and Masters ...

👤 Ts. Ahmad Hadinata bin Fauzi
   Profile (first 200 chars): Ts. Hj. Ahmad Hadinata bin Fauzi is presently working as a Senior Lecturer in Network Computing Program at Faculty of Computer Science and Information Technology in Universiti Malaysia Sarawak (UNIMAS...


In [33]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import json, time

In [34]:
MODEL_NAME = 'all-MiniLM-L6-v2'

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded")

Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [12]:
df = pd.read_csv('lecturers_clean_2.csv')
print(f"Embedding {len(df)} lecturer profiles...")

Embedding 65 lecturer profiles...


In [13]:
start = time.time()
profiles = df['profile'].tolist()

embeddings = model.encode(
    profiles,
    batch_size=16,          # process 16 at a time
    show_progress_bar=True, # shows a nice progress bar
    normalize_embeddings=True  # normalise → cosine sim = dot product
)

elapsed = time.time() - start
print(f"\n✅ Done! {len(embeddings)} embeddings in {elapsed:.1f}s")
print(f"   Embedding shape: {embeddings.shape}")  # (65, 384)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Done! 65 embeddings in 19.9s
   Embedding shape: (65, 384)


In [15]:
np.save('embeddings.npy', embeddings)
print("✅ Saved: embeddings.npy")

# ── Quick sanity check ────────────────────────────────────────────────
print("\n--- Sanity Check: Most similar lecturers to lecturer #0 ---")
from numpy.linalg import norm

target = embeddings[0]
sims = embeddings @ target  # dot product (already normalised = cosine sim)
top5 = np.argsort(sims)[::-1][:5]
for i in top5:
    print(f"  [{sims[i]:.4f}] {df.iloc[i]['name']}")

✅ Saved: embeddings.npy

--- Sanity Check: Most similar lecturers to lecturer #0 ---
  [1.0000] Ts. Abdul Rahman bin Mat
  [0.7205] Encik Mohamad Johan bin Ahmad Khiri
  [0.7115] Puan Norazian Binti Mohamad Hamdan
  [0.7006] Dr Muhammad Asyraf bin Khairuddin
  [0.6703] Ts. Nurfauza bt Jali


In [ ]:
import pandas as pd
import numpy as np
import vector_store
import ast, json

In [17]:
df = pd.read_csv('lecturers_clean.csv')
embeddings = np.load('embeddings.npy')
print(f"Loaded {len(df)} lecturers + embeddings shape {embeddings.shape}")

Loaded 65 lecturers + embeddings shape (65, 384)


In [29]:
import re
import pandas as pd

def extract_research_sentences(text):
    # ✅ Handle NaN FIRST
    if pd.isna(text) or str(text).lower() == "nan":
        return ""
    
    text = str(text)

    sentences = re.split(r'(?<=[.!?])\s+', text)

    keywords = ["research", "interest", "expertise", "focus", "work"]

    selected = [
        s for s in sentences
        if any(k in s.lower() for k in keywords)
    ]

    return " ".join(selected)

In [30]:
def chunk_list(items, chunk_size=5):
    return [items[i:i+chunk_size] for i in range(0, len(items), chunk_size)]

In [46]:
documents = []
metadatas = []
ids = []

for _, row in df.iterrows():
    staff_id = str(row["staff_no"])
    name = row["name"]

    # -------- 1. Summary chunk --------
    summary = clean(extract_research_sentences(row.get("summary", "")))
    
    if summary:
        documents.append(f"Expertise: {summary}")
        metadatas.append({
            "name": name,
            "staff_no": staff_id,
            "type": "summary"
        })
        ids.append(f"{staff_id}_summary")

    # -------- 2. Publications --------
    titles_all = []

    for col in ['articles', 'proceedings']:
        text = clean(row.get(col, ''))

        if text:
            titles = [t.strip() for t in text.split(";") if t.strip()]
            titles_all.extend(titles)

    # remove duplicates
    titles_all = list(dict.fromkeys(titles_all))

    # -------- 3. Chunk publications --------
    chunks = chunk_list(titles_all, chunk_size=5)

    for i, chunk in enumerate(chunks):
        documents.append("Research topics: " + "; ".join(chunk))
        metadatas.append({
            "name": name,
            "staff_no": staff_id,
            "summary": row.get("summary", ""),
            "type": "publication_chunk",
            "num_articles": int(row.get("num_articles", 0) or 0),
            "num_proceedings": int(row.get("num_proceedings", 0) or 0),
            "email": row.get("email"),
            "google_scholar": row.get("google_scholar"),
            "img_url": row.get("img_url")
        })
        ids.append(f"{staff_id}_chunk_{i}")

In [47]:
import chromadb
from config import CHROMA_PATH

client = chromadb.PersistentClient(path=CHROMA_PATH)

client.delete_collection("lecturers_chunked")

collection = client.create_collection("lecturers_chunked")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

In [50]:
print(data["metadatas"][0].keys())

dict_keys(['staff_no', 'name', 'type'])


In [35]:
start = time.time()

embeddings = model.encode(documents, normalize_embeddings=True)

elapsed = time.time() - start
print(f"\n✅ Done! {len(embeddings)} embeddings in {elapsed:.1f}s")
print(f"   Embedding shape: {embeddings.shape}")  # (65, 384)


✅ Done! 455 embeddings in 6.7s
   Embedding shape: (455, 384)


In [38]:
import chromadb
from config import CHROMA_PATH

client = chromadb.PersistentClient(path=CHROMA_PATH)

# delete old if exists (optional but recommended during testing)
try:
    client.delete_collection("lecturers_chunked")
except:
    pass

collection = client.create_collection("lecturers_chunked")

In [39]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

In [40]:
print("Total vectors:", collection.count())

Total vectors: 455


In [44]:
test_query = "Sign language detection"

results = collection.query(
    query_texts=[test_query],
    n_results=3
)

In [48]:
for i in range(len(results["documents"][0])):
    print(f"Result {i+1}")
    print("Document:", results["documents"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Distance:", results["distances"][0][i])
    print("-" * 50)

Result 1
Document: Research topics: Language Modelling for a Low-Resource Language in Sarawak, Malaysia; Unsupervised segmentation of action segments in egocentric videos using gaze; Merging of native and non-native speech for low-resource accented ASR; Using resources from a closely-related language to develop ASR for a very under-resourced language: A case study for iban; Using closely-related language to build an ASR for a very under-resourced language: Iban
Metadata: {'email': 'sjsflora@unimas.my', 'staff_no': '1319', 'type': 'publication_chunk', 'name': 'Prof Madya Ts Dr Sarah Flora Anak Samson Juan', 'google_scholar': 'https://scholar.google.fr/citations?user=oMLD0UYAAAAJ&hl=en', 'img_url': 'https://directory.unimas.my/assets/images/staffpic/1319.jpg'}
Distance: 1.2784470319747925
--------------------------------------------------
Result 2
Document: Research topics: Malaysian Sign Language Recognition Using 3D Hand Pose Estimation; Do You Know My Name? Learning Mandarin through G

In [23]:
df = pd.read_csv('lecturers_clean_2.csv')
print(f"Loaded {len(df)} lecturers")

Loaded 65 lecturers


In [51]:
client.delete_collection("lecturers_chunked")

In [52]:
collection = client.create_collection("lecturers_chunked")

In [53]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

In [54]:
data = collection.get(limit=1)
print(data["metadatas"][0].keys())

dict_keys(['name', 'type', 'staff_no'])


In [2]:
import chromadb
from config import CHROMA_PATH

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection("lecturers_chunked")

# Check collection metadata (confirms the distance metric)
print("Collection metadata:", collection.metadata)

# Get a specific document and its embedding vector
result = collection.get(
    ids=["1331_summary"],  # replace with a real ID e.g. "123_summary"
    include=["embeddings", "documents", "metadatas"]
)

print("Document:", result["documents"][0])
print("Embedding length:", len(result["embeddings"][0]))
print("First 10 values:", result["embeddings"][0][:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection metadata: None
Document: Expertise: Her research interest is in operations research. She is currently working in combinatorial problems, focusing educational timetabling, vehicle routing and metaheuristic algorithm.
Embedding length: 384
First 10 values: [ 0.00955549 -0.02194205  0.01434525  0.03304169 -0.09999632 -0.06570611
  0.06770884  0.04105102 -0.13304061  0.01444931]


In [3]:
import pandas as pd

df = pd.read_csv("lecturers_clean_2.csv")

df["profile_url"] = df["staff_no"].apply(lambda x: f"https://expert.unimas.my/profile/{x}")

df.to_csv("lecturers_clean_2.csv", index=False)

print(df[["staff_no", "profile_url"]].head(10))

  staff_no                             profile_url
0      861    https://expert.unimas.my/profile/861
1    K0537  https://expert.unimas.my/profile/K0537
2      858    https://expert.unimas.my/profile/858
3     1330   https://expert.unimas.my/profile/1330
4      491    https://expert.unimas.my/profile/491
5     1150   https://expert.unimas.my/profile/1150
6     1089   https://expert.unimas.my/profile/1089
7     3159   https://expert.unimas.my/profile/3159
8     1379   https://expert.unimas.my/profile/1379
9     1652   https://expert.unimas.my/profile/1652


In [4]:
import chromadb
from config import CHROMA_PATH

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection("lecturers_chunked")

# Check collection metadata (confirms the distance metric)
print("Collection metadata:", collection.metadata)

# Get a specific document and its embedding vector
result = collection.get(
    ids=["1331_summary"],  # replace with a real ID e.g. "123_summary"
    include=["embeddings", "documents", "metadatas"]
)

print("Document:", result["documents"][0])
print("Embedding length:", len(result["embeddings"][0]))
print("First 10 values:", result["embeddings"][0][:10])

Collection metadata: {'hnsw:space': 'cosine'}
Document: Profesor Madya Dr. Sze San Nah expertise: She received her PhD in Applied Mathematics specialised in Operations Management from University of Sydney, Australia. Her research interest is in operations research. She is currently working in combinatorial problems, focusing educational timetabling, vehicle routing and metaheuristic algorithm.
Embedding length: 768
First 10 values: [ 0.02783286 -0.02124875 -0.02655501 -0.00205461 -0.01622037 -0.0679042
  0.04440898  0.00185227 -0.0786983  -0.01966983]
